# 🚀 Train CodeMaster-CRNN on IAM Handwriting Dataset (Google Colab GPU)

This notebook trains and fine-tunes the **PyTorch CRNN + CTC** handwriting recognition model on the **IAM Handwriting Word Database** using free Google Colab GPU acceleration (T4/A100).

### Pipeline Overview:
1. **Hardware Check**: Validates GPU acceleration (`torch.cuda.is_available()`).
2. **Dataset Setup**: Downloads and parses the IAM Words dataset (`words.txt` and images).
3. **Preprocessing**: Strict $(32, 128)$ aspect-ratio padding with background auto-inversion.
4. **Model Architecture**: 4-stage CNN + Map-to-Sequence + 2-layer BiLSTM + 77-class Linear Head.
5. **Training Engine**: `nn.CTCLoss`, Adam optimizer, and `ReduceLROnPlateau` scheduler.
6. **Evaluation**: Computes Character Error Rate (CER) and Word Error Rate (WER) per epoch on held-out validation and test splits.
7. **Export**: Saves and downloads `best_model_gpu.pth` and `vocab.json` directly to your local computer.

## 1. Check GPU Runtime Acceleration

In [ ]:
import torch
import os

print("PyTorch Version:", torch.__version__)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU Active: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected. In Colab, go to Runtime > Change runtime type > Select T4 GPU.")

## 2. Install Dependencies & Download IAM Dataset

In [ ]:
!pip install -q opencv-python matplotlib tqdm Levenshtein

# Clone the CodeMaster-AJ repository to obtain words.txt and the complete IAM dataset
!git clone https://github.com/CodeMaster-AJ/handwritten-text-recognition.git repo_data

DATA_DIR = "repo_data/iam_words"
TXT_PATH = os.path.join(DATA_DIR, "words.txt")
IMG_DIR = os.path.join(DATA_DIR, "words")

assert os.path.exists(TXT_PATH), f"Annotation file not found at {TXT_PATH}!"
assert os.path.exists(IMG_DIR), f"Images directory not found at {IMG_DIR}!"
print(f"✅ Dataset verified at {DATA_DIR}")

## 3. Dataset Parsing & Character Mapping

In [ ]:
import json

def parse_iam_dataset(txt_path, img_dir):
    samples = []
    chars = set()
    with open(txt_path, "r", encoding="latin1") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) < 9 or parts[1] != "ok":
                continue
            file_id = parts[0]
            word_text = parts[-1]
            id_parts = file_id.split("-")
            sub1 = id_parts[0]
            sub2 = f"{id_parts[0]}-{id_parts[1]}"
            nested_img_path = os.path.join(img_dir, sub1, sub2, f"{file_id}.png")
            if os.path.exists(nested_img_path):
                samples.append((nested_img_path, word_text))
                for char in word_text:
                    chars.add(char)
    alphabet = sorted(list(chars))
    return samples, alphabet

samples, alphabet = parse_iam_dataset(TXT_PATH, IMG_DIR)
print(f"Found {len(samples)} valid images with {len(alphabet)} unique characters.")

# Index 0 is reserved for the CTC blank token
char_to_num = {char: idx + 1 for idx, char in enumerate(alphabet)}
num_to_char = {idx + 1: char for idx, char in enumerate(alphabet)}
num_classes = len(alphabet) + 1  # 77 classes

# Save vocabulary mapping
vocab_info = {
    "alphabet": "".join(alphabet),
    "num_classes": num_classes,
    "blank_index": 0,
    "char_to_num": char_to_num,
    "num_to_char": {str(k): v for k, v in num_to_char.items()},
}
with open("vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_info, f, indent=2)
print(f"✅ Vocabulary saved with {num_classes} classes (Blank = 0).")

## 4. Dataset Splits & PyTorch DataLoaders

Splits: **90% Training**, **5% Validation**, and **5% Held-Out Testing**.

In [ ]:
import cv2
import numpy as np
from torch.utils.data import Dataset, DataLoader

class IAMWordDataset(Dataset):
    def __init__(self, samples, char_to_num, img_w=128, img_h=32):
        self.samples = samples
        self.char_to_num = char_to_num
        self.img_w = img_w
        self.img_h = img_h

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, word = self.samples[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.ones((self.img_h, self.img_w), dtype=np.uint8) * 255

        h, w = img.shape
        f = min(self.img_w / max(1, w), self.img_h / max(1, h))
        new_w, new_h = max(1, int(w * f)), max(1, int(h * f))
        img_resized = cv2.resize(img, (new_w, new_h))

        # White canvas padding (value 255)
        canvas = np.ones((self.img_h, self.img_w), dtype=np.uint8) * 255
        canvas[0:new_h, 0:new_w] = img_resized

        # Check corner shadows / polarity
        corners = [canvas[0, 0], canvas[0, -1], canvas[-1, 0], canvas[-1, -1]]
        if np.mean(corners) < 128:
            canvas = 255 - canvas

        img_normalized = canvas.astype(np.float32) / 255.0
        img_tensor = np.expand_dims(img_normalized, axis=0)

        label_list = [self.char_to_num[c] for c in word if c in self.char_to_num]
        if not label_list:
            label_list = [self.char_to_num.get('a', 1)]

        return torch.tensor(img_tensor, dtype=torch.float32), label_list

def ctc_collate(batch):
    images = []
    targets = []
    target_lengths = []
    for img, labels in batch:
        images.append(img)
        targets.extend(labels)
        target_lengths.append(len(labels))
    return (
        torch.stack(images, dim=0),
        torch.tensor(targets, dtype=torch.long),
        torch.tensor(target_lengths, dtype=torch.long),
    )

np.random.seed(42)
shuffled_samples = list(samples)
np.random.shuffle(shuffled_samples)

n_total = len(shuffled_samples)
n_train = int(n_total * 0.90)
n_val = int(n_total * 0.05)

train_set = shuffled_samples[:n_train]
val_set = shuffled_samples[n_train : n_train + n_val]
test_set = shuffled_samples[n_train + n_val:]

print(f"Splits: Train={len(train_set)}, Val={len(val_set)}, Test={len(test_set)}")

train_loader = DataLoader(IAMWordDataset(train_set, char_to_num), batch_size=64, shuffle=True, collate_fn=ctc_collate, num_workers=2)
val_loader = DataLoader(IAMWordDataset(val_set, char_to_num), batch_size=64, shuffle=False, collate_fn=ctc_collate, num_workers=2)
test_loader = DataLoader(IAMWordDataset(test_set, char_to_num), batch_size=64, shuffle=False, collate_fn=ctc_collate, num_workers=2)

## 5. Model Architecture (CodeMaster CRNN)

In [ ]:
import torch.nn as nn

class CRNN(nn.Module):
    def __init__(self, num_classes):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 1))
        )
        self.linear = nn.Linear(256 * 4, 128)
        self.lstm = nn.LSTM(128, 128, bidirectional=True, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x):
        features = self.cnn(x)
        features = features.permute(0, 3, 1, 2)
        b, t, c, h = features.size()
        features = features.contiguous().view(b, t, c * h)
        seq_out = self.linear(features)
        seq_out, _ = self.lstm(seq_out)
        output_logits = self.fc(seq_out)
        return output_logits

model = CRNN(num_classes).to(device)

# Optionally load the initial 5-epoch baseline checkpoint to start fine-tuning
init_ckpt_path = "repo_data/saved_models/best_model.pth"
if os.path.exists(init_ckpt_path):
    print(f"Initializing weights from baseline checkpoint: {init_ckpt_path}")
    model.load_state_dict(torch.load(init_ckpt_path, map_location=device))
else:
    print("Training from scratch.")

print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Metric Functions: CTC Greedy Decoder, CER & WER

In [ ]:
def greedy_decode(pred_indexes, num_to_char):
    decoded = ""
    prev = None
    for token in pred_indexes:
        if token != 0 and token != prev:
            decoded += num_to_char.get(token, "")
        prev = token
    return decoded

def levenshtein(s1, s2):
    d = np.zeros((len(s1) + 1, len(s2) + 1), dtype=int)
    for i in range(len(s1) + 1): d[i, 0] = i
    for j in range(len(s2) + 1): d[0, j] = j
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)
    return d[len(s1), len(s2)]

def compute_cer_wer(refs, hyps):
    total_char_dist = 0
    total_char_len = 0
    word_errors = 0
    for r, h in zip(refs, hyps):
        total_char_dist += levenshtein(r.lower(), h.lower())
        total_char_len += max(1, len(r))
        if r.lower() != h.lower():
            word_errors += 1
    cer = total_char_dist / max(1, total_char_len)
    wer = word_errors / max(1, len(refs))
    return cer, wer

## 7. Deep GPU Training Loop (50 Epochs with Early Stopping)

In [ ]:
import time
from tqdm import tqdm

EPOCHS = 50
LEARNING_RATE = 0.0005
BEST_MODEL_PATH = "best_model_gpu.pth"

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "val_cer": [], "val_wer": []}

print("Starting GPU Training Run...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0

    for images, targets, target_lengths in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        outputs_ctc = outputs.permute(1, 0, 2)

        b_size = images.size(0)
        input_lengths = torch.full((b_size,), 32, dtype=torch.long, device=device)

        loss = criterion(outputs_ctc, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss += loss.item() * b_size

    epoch_train_loss = train_loss / len(train_set)

    # Validation Loop
    model.eval()
    val_loss = 0.0
    all_refs = []
    all_hyps = []

    with torch.no_grad():
        for images, targets, target_lengths in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            target_lengths = target_lengths.to(device)

            outputs = model(images)
            outputs_ctc = outputs.permute(1, 0, 2)
            b_size = images.size(0)
            input_lengths = torch.full((b_size,), 32, dtype=torch.long, device=device)

            loss = criterion(outputs_ctc, targets, input_lengths, target_lengths)
            val_loss += loss.item() * b_size

            pred_indexes = torch.argmax(outputs, dim=2).cpu().numpy()
            targets_arr = targets.cpu().numpy()
            lens_arr = target_lengths.cpu().numpy()

            offset = 0
            for b in range(b_size):
                l = lens_arr[b]
                ref_text = "".join(num_to_char.get(c, "") for c in targets_arr[offset : offset + l])
                offset += l
                hyp_text = greedy_decode(pred_indexes[b], num_to_char)
                all_refs.append(ref_text)
                all_hyps.append(hyp_text)

    epoch_val_loss = val_loss / len(val_set)
    val_cer, val_wer = compute_cer_wer(all_refs, all_hyps)
    elapsed = time.time() - t0

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_cer"].append(val_cer)
    history["val_wer"].append(val_wer)

    print(f"Epoch {epoch:2d}/{EPOCHS:2d} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val CER: {val_cer:.2%} | Val WER: {val_wer:.2%} | Time: {elapsed:.1f}s")
    scheduler.step(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  --> 💾 Saved new best model to {BEST_MODEL_PATH}")

## 8. Final Evaluation on Held-Out Test Split

In [ ]:
# Load best checkpoint for test evaluation
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

test_refs = []
test_hyps = []

with torch.no_grad():
    for images, targets, target_lengths in test_loader:
        images = images.to(device)
        outputs = model(images)
        pred_indexes = torch.argmax(outputs, dim=2).cpu().numpy()
        targets_arr = targets.numpy()
        lens_arr = target_lengths.numpy()
        offset = 0
        for b in range(images.size(0)):
            l = lens_arr[b]
            ref_text = "".join(num_to_char.get(c, "") for c in targets_arr[offset : offset + l])
            offset += l
            hyp_text = greedy_decode(pred_indexes[b], num_to_char)
            test_refs.append(ref_text)
            test_hyps.append(hyp_text)

test_cer, test_wer = compute_cer_wer(test_refs, test_hyps)
print(f"\n================ FINAL TEST BENCHMARK ================")
print(f"Test Set Size: {len(test_refs)} word images")
print(f"Test Character Error Rate (CER): {test_cer:.2%}")
print(f"Test Word Error Rate (WER):      {test_wer:.2%}")
print(f"Test Word Accuracy:              {1.0 - test_wer:.2%}")
print("=======================================================")

print("\n--- Sample Predictions on Test Data ---")
for i in range(10):
    print(f"Ground Truth: '{test_refs[i]:15s}' | Predicted: '{test_hyps[i]:15s}' | {'MATCH' if test_refs[i].lower() == test_hyps[i].lower() else 'DIFF'}")

## 9. Test Any Custom Handwriting Image

In [ ]:
def transcribe_word(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return f"File not found: {image_path}"
    h, w = img.shape
    f = min(128 / w, 32 / h)
    new_w, new_h = max(1, int(w * f)), max(1, int(h * f))
    img_res = cv2.resize(img, (new_w, new_h))
    canvas = np.ones((32, 128), dtype=np.uint8) * 255
    canvas[0:new_h, 0:new_w] = img_res
    if np.mean([canvas[0, 0], canvas[0, -1], canvas[-1, 0], canvas[-1, -1]]) < 128:
        canvas = 255 - canvas
    tensor_in = torch.tensor(canvas.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        preds = model(tensor_in)
        indexes = torch.argmax(preds, dim=2).squeeze(0).cpu().numpy()
    return greedy_decode(indexes, num_to_char)

# Test on repository sample image
sample_test = "repo_data/test_images/my_word.png"
print(f"Prediction for {sample_test}: '{transcribe_word(sample_test)}'")

## 10. Download Checkpoint & Vocabulary to Local Project

In [ ]:
from google.colab import files

print("Downloading best_model_gpu.pth and vocab.json...")
files.download(BEST_MODEL_PATH)
files.download("vocab.json")
print("✅ Place best_model_gpu.pth in models/checkpoints/ inside your local project.")